<a href="https://colab.research.google.com/github/miray7yuce/quadcopter-rl-copilot/blob/main/notebooks/quadcopter_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q stable-baselines3 gymnasium
!pip install -q jsbsim==1.2.4
!pip install -q pyyaml
!pip install -q optuna

import jsbsim
print(jsbsim.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 20.9 MB/s eta 0:00:00
1.2.4


In [2]:
from google.colab import userdata
import os

USER  = "miray7yuce"
REPO  = "quadcopter-rl-copilot"
TOKEN = userdata.get('GH_TOKEN')

!git config --global user.email "miray7yuce@gmail.com"
!git config --global user.name "miray7yuce"

os.environ['REMOTE'] = f"https://{TOKEN}@github.com/{USER}/{REPO}.git"
!rm -rf /content/repo
!git clone -q $REMOTE /content/repo
!ls -a /content/repo

.			 .gitignore		    README.md
..			 notebooks		    requirements.txt
configs			 ppo_final.acmi		    runs
f450-drone-framestl.stl  ppo_telemetry.csv	    sac_final.acmi
.git			 ppo_vs_sac_comparison.png  src


In [3]:
import os
os.chdir('/content')
print(os.getcwd())

/content


In [4]:
!pip freeze | grep -iE "^(jsbsim|stable-baselines3|gymnasium|torch|numpy)=" > /content/repo/requirements.txt
!cat /content/repo/requirements.txt

gymnasium==1.3.0
jsbsim==1.2.4
numpy==2.1.3


In [5]:
import os, sys

BASE = "/content/repo"

for d in ["src/drone_rl/envs", "src/drone_rl/utils", "configs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

for p in ["src/drone_rl", "src/drone_rl/envs", "src/drone_rl/utils"]:
    open(f"{BASE}/{p}/__init__.py", "a").close()

with open(f"{BASE}/.gitignore", "w") as f:
    f.write("__pycache__/\n*.zip\n*.pkl\nlogs/\nruns/\n.ipynb_checkpoints/\n")

sys.path.insert(0, f"{BASE}/src")

!find /content/repo -not -path '*/.git/*' -type f | sort

os.environ['PYTHONPATH'] = f"{BASE}/src"

/content/repo/configs/ppo_flight.yaml
/content/repo/configs/ppo_hover_customnet.yaml
/content/repo/configs/ppo_hover.yaml
/content/repo/f450-drone-framestl.stl
/content/repo/.gitignore
/content/repo/notebooks/quadcopter_rl.ipynb
/content/repo/ppo_final.acmi
/content/repo/ppo_telemetry.csv
/content/repo/ppo_vs_sac_comparison.png
/content/repo/README.md
/content/repo/requirements.txt
/content/repo/runs/hover_v1/model.zip
/content/repo/runs/hover_v1/vecnormalize.pkl
/content/repo/sac_final.acmi
/content/repo/src/drone_rl/acmi_writer.py
/content/repo/src/drone_rl/config.py
/content/repo/src/drone_rl/env_factory.py
/content/repo/src/drone_rl/envs/f450_env.py
/content/repo/src/drone_rl/envs/f450_flight_env.py
/content/repo/src/drone_rl/envs/__init__.py
/content/repo/src/drone_rl/evaluate.py
/content/repo/src/drone_rl/__init__.py
/content/repo/src/drone_rl/train.py
/content/repo/src/drone_rl/tune.py
/content/repo/src/drone_rl/utils/__init__.py
/content/repo/src/drone_rl/utils/units.py
/conten

In [6]:
%%writefile /content/repo/src/drone_rl/utils/units.py
"""Birim donusumleri. JSBSim emperyal birim kullanir."""

FT2M = 0.3048
M2FT = 1.0 / FT2M

def ft_to_m(x):
    return x * FT2M

def m_to_ft(x):
    return x * M2FT

Overwriting /content/repo/src/drone_rl/utils/units.py


In [7]:
%%writefile /content/repo/src/drone_rl/envs/f450_flight_env.py
"""F450 quadcopter icin hedef irtifa + hedef yon (heading) takip gorevi.

f450_env.py'deki F450HoverEnv'e HIC dokunulmadan, ayri bir env olarak
eklenmistir. Heading burada 'hareket yonu' (course over ground) anlamina
gelir, burun yonu (yaw) degil - yani drone burnu farkli yone bakarken bile
komut edilen yonde ilerleyebilir.

DUZELTME (v2):
- Drone artik hedef irtifanin ALTINDAN spawn oluyor (once bir tirmanma
  gerceklesiyor), hedefin hemen yaninda degil.
- Hedef irtifaya ulasip orada bir sure kalinca episode BASARIYLA
  sonlaniyor (terminated=True + success_bonus), sadece sure dolunca
  degil.
- Heading/hiz odulu, irtifa ilerlemesiyle orantili agirliklandiriliyor:
  drone hala tirmanirken yon odulu zayif, hedefe yaklastikca guclenir.
  Boylece 'once yuksel, sonra yavasca hedef yone don' seklinde bir
  yay/spiral davranisi ortaya cikmasi tesvik edilir.
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import jsbsim


class F450FlightEnv(gym.Env):
    """JSBSim F450 modeli uzerinde: hedef irtifaya TIRMAN, tirmanirken
    yavasca hedef yone (dunya cercevesinde, pusula konvansiyonu:
    0=Kuzey, 90=Dogu) don, hedef irtifaya ulasinca episode'u basariyla
    bitir (reset) gorevi.

    Her episode'da target_altitude VE target_heading RASTGELE secilir
    (F450HoverEnv'de target_altitude sabitti). Model bu ikisini gozlem
    olarak alir (alt_err + sin/cos(heading)), yani 'goal-conditioned' bir
    politika ogrenir.
    """

    metadata = {"render_modes": []}

    def __init__(
        self,
        target_altitude_min_ft=20.0,
        target_altitude_max_ft=45.0,
        target_speed_fps=6.0,
        episode_seconds=60.0,
        physics_hz=240,
        control_hz=20,
        hover_throttle=0.420,
        throttle_range=0.25,
        reward_alt_weight=0.10,
        reward_heading_weight=0.08,
        reward_tilt_weight=0.05,
        reward_spin_weight=0.10,
        reward_jerk_weight=0.05,
        crash_penalty=50.0,
        crash_min_alt_ft=1.0,
        crash_max_alt_offset_ft=60.0,
        crash_max_tilt_rad=1.0,
        # --- YENI parametreler ---
        altitude_start_offset_ft=25.0,
        altitude_start_jitter_ft=2.0,
        success_alt_tol_ft=1.5,
        success_hold_seconds=1.0,
        success_bonus=20.0,
    ):
        super().__init__()

        physics_hz = int(physics_hz)
        control_hz = int(control_hz)
        if physics_hz <= 0 or control_hz <= 0:
            raise ValueError("physics_hz ve control_hz pozitif olmali")
        if physics_hz % control_hz != 0:
            raise ValueError(
                f"physics_hz ({physics_hz}) control_hz'e ({control_hz}) tam "
                "bolunmeli. Or: 240/20=12 OK, 240/50 HATALI."
            )

        # Gozlem: alt_err, hdot, along_track, cross_track, roll, pitch,
        #         p, q, r, sin(heading), cos(heading), prev_action(4) = 15
        self.action_space = spaces.Box(-1.0, 1.0, shape=(4,), dtype=np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(15,), dtype=np.float32)

        self.target_altitude_min_ft = target_altitude_min_ft
        self.target_altitude_max_ft = target_altitude_max_ft
        self.target_speed_fps = target_speed_fps

        self.physics_hz = physics_hz
        self.physics_dt = 1.0 / physics_hz
        self.control_hz = control_hz
        self.substeps = physics_hz // control_hz
        self.max_steps = int(episode_seconds * control_hz)

        self.hover_throttle = hover_throttle
        self.throttle_range = throttle_range

        self.reward_alt_weight = reward_alt_weight
        self.reward_heading_weight = reward_heading_weight
        self.reward_tilt_weight = reward_tilt_weight
        self.reward_spin_weight = reward_spin_weight
        self.reward_jerk_weight = reward_jerk_weight
        self.crash_penalty = crash_penalty
        self.crash_min_alt_ft = crash_min_alt_ft
        self.crash_max_alt_offset_ft = crash_max_alt_offset_ft
        self.crash_max_tilt_rad = crash_max_tilt_rad

        # --- YENI: baslangic offseti ve basari (success) ayarlari ---
        self.altitude_start_offset_ft = altitude_start_offset_ft
        self.altitude_start_jitter_ft = altitude_start_jitter_ft
        self.success_alt_tol_ft = success_alt_tol_ft
        self.success_hold_steps = max(1, int(success_hold_seconds * control_hz))
        self.success_bonus = success_bonus
        self._success_counter = 0
        # Ilerleme (progress) hesaplamak icin: episode basindaki irtifa farki
        self._initial_alt_err_ft = 1.0  # reset()'te gercek degerle guncellenir

        # Her episode'da rastgele secilecek hedefler - reset()'te doldurulur
        self.target_altitude = (target_altitude_min_ft + target_altitude_max_ft) / 2.0
        self.target_heading = 0.0

        self.fdm = jsbsim.FGFDMExec(None)
        self.fdm.set_debug_level(0)
        if not self.fdm.load_model("F450"):
            raise RuntimeError("F450 modeli yuklenemedi")
        self.fdm.set_dt(self.physics_dt)

        self.step_count = 0
        self.prev_action = np.zeros(4, dtype=np.float32)

    @property
    def control_dt(self):
        return self.substeps * self.physics_dt

    def _apply_initial_conditions(self):
        # DUZELTME: drone artik hedef irtifanin ALTINDAN basliyor, hemen
        # yaninda degil - boylece gercek bir tirmanma gerceklesir.
        jitter = self.np_random.uniform(
            -self.altitude_start_jitter_ft, self.altitude_start_jitter_ft
        )
        h0 = self.target_altitude - self.altitude_start_offset_ft + jitter
        # Yerden/crash siniri altina dusmesin diye guvenlik payi birak.
        h0 = max(h0, self.crash_min_alt_ft + 3.0)
        self.fdm["ic/h-agl-ft"] = h0

        self.fdm["ic/u-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/v-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/w-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/phi-rad"] = self.np_random.uniform(-0.05, 0.05)
        self.fdm["ic/theta-rad"] = self.np_random.uniform(-0.05, 0.05)
        self.fdm["ic/psi-true-rad"] = 0.0

        return abs(h0 - self.target_altitude)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # her episode'da hedef irtifa VE hedef yon rastgele secilir
        self.target_altitude = self.np_random.uniform(
            self.target_altitude_min_ft, self.target_altitude_max_ft
        )
        self.target_heading = self.np_random.uniform(0.0, 2.0 * np.pi)

        initial_alt_err = self._apply_initial_conditions()
        self._initial_alt_err_ft = max(initial_alt_err, 1e-3)
        self.fdm.run_ic()

        for i in range(4):
            self.fdm[f"propulsion/engine[{i}]/set-running"] = 1
        self.fdm["fcs/ScasEngage"] = 0

        for i in range(4):
            self.fdm[f"fcs/throttle-cmd-norm[{i}]"] = self.hover_throttle

        self.step_count = 0
        self.prev_action = np.zeros(4, dtype=np.float32)
        self._success_counter = 0

        return self._get_obs(), {}

    def _world_frame_velocity(self):
        """Govde-cercevesi (body-frame) u,v hizlarini, mevcut yaw (psi)
        acisini kullanarak dunya-cercevesi (kuzey, dogu) hizlarina cevirir.
        Yaw zamanla suruklenirse bile bu donusum otomatik dogru kalir."""
        f = self.fdm
        u = f["velocities/u-fps"]
        v = f["velocities/v-fps"]
        psi = f["attitude/psi-rad"]
        north_vel = u * np.cos(psi) - v * np.sin(psi)
        east_vel = u * np.sin(psi) + v * np.cos(psi)
        return north_vel, east_vel

    def _along_cross_track(self):
        """Dunya-cercevesi hizi, hedef yone gore 'ileri (along-track)' ve
        'yana kayma (cross-track)' bilesenlerine ayirir. Aci hesabi
        (atan2) kullanmadigimiz icin dusuk hizda tekillik/gurultu olmaz."""
        north_vel, east_vel = self._world_frame_velocity()
        h = self.target_heading
        along = north_vel * np.cos(h) + east_vel * np.sin(h)
        cross = -north_vel * np.sin(h) + east_vel * np.cos(h)
        return along, cross

    def _climb_progress(self, alt_err_ft):
        """0 (hala baslangic irtifasinda) -> 1 (hedef irtifaya ulasti)
        arasinda bir ilerleme skoru. Heading odulunu bununla carparak
        drone once yukselirken yon odulunu zayif tutuyoruz, hedefe
        yaklastikca guclendiriyoruz -> 'yay/spiral' davranisi."""
        progress = 1.0 - (alt_err_ft / self._initial_alt_err_ft)
        return float(np.clip(progress, 0.0, 1.0))

    def _get_obs(self):
        f = self.fdm
        alt_err = (f["position/h-agl-ft"] - self.target_altitude) / 10.0
        hdot = f["velocities/h-dot-fps"] / 10.0

        along, cross = self._along_cross_track()
        along_n = along / 10.0
        cross_n = cross / 10.0

        roll = f["attitude/phi-rad"]
        pitch = f["attitude/theta-rad"]
        p = f["velocities/p-rad_sec"] / 5.0
        q = f["velocities/q-rad_sec"] / 5.0
        r = f["velocities/r-rad_sec"] / 5.0

        sin_h = np.sin(self.target_heading)
        cos_h = np.cos(self.target_heading)

        return np.array(
            [alt_err, hdot, along_n, cross_n, roll, pitch, p, q, r,
             sin_h, cos_h, *self.prev_action],
            dtype=np.float32,
        )

    def _get_telemetry(self, crashed, reached):
        f = self.fdm
        alt_agl_ft = float(f["position/h-agl-ft"])
        along, cross = self._along_cross_track()
        return {
            "alt_ft": alt_agl_ft,
            "alt_sl_ft": float(f["position/h-sl-ft"]),
            "alt_err_ft": abs(alt_agl_ft - self.target_altitude),
            "lat_deg": float(f["position/lat-geod-deg"]),
            "lon_deg": float(f["position/long-gc-deg"]),
            "x_m": float(f["position/distance-from-start-lon-mt"]),
            "y_m": float(f["position/distance-from-start-lat-mt"]),
            "roll_rad": float(f["attitude/phi-rad"]),
            "pitch_rad": float(f["attitude/theta-rad"]),
            "yaw_rad": float(f["attitude/psi-rad"]),
            "target_heading_rad": float(self.target_heading),
            "along_track_fps": float(along),
            "cross_track_fps": float(cross),
            "target_speed_fps": float(self.target_speed_fps),
            "crashed": bool(crashed),
            "reached_target": bool(reached),
        }

    def _is_crashed(self):
        alt = self.fdm["position/h-agl-ft"]
        return (
            alt < self.crash_min_alt_ft
            or alt > self.target_altitude + self.crash_max_alt_offset_ft
            or abs(self.fdm["attitude/phi-rad"]) > self.crash_max_tilt_rad
            or abs(self.fdm["attitude/theta-rad"]) > self.crash_max_tilt_rad
        )

    def step(self, action):
        action = np.asarray(action, dtype=np.float32).reshape(4)

        throttles = np.clip(
            self.hover_throttle + action * self.throttle_range, 0.0, 1.0
        )

        for _ in range(self.substeps):
            for i in range(4):
                self.fdm[f"fcs/throttle-cmd-norm[{i}]"] = float(throttles[i])
            self.fdm.run()

        self.step_count += 1
        obs = self._get_obs()

        alt_err_ft = abs(self.fdm["position/h-agl-ft"] - self.target_altitude)
        along, cross = self._along_cross_track()

        # DUZELTME: heading/hiz odulu, tirmanma ilerlemesiyle carpiliyor.
        # Drone hala baslangic irtifasindaysa (progress~0) yon odulu
        # zayif; hedefe yaklastikca (progress->1) guclenir. Boylece
        # ajan once dikey harekete, sonra yatay harekete agirlik verir.
        progress = self._climb_progress(alt_err_ft)
        heading_err = abs(self.target_speed_fps - along) + abs(cross)

        tilt = abs(self.fdm["attitude/phi-rad"]) + abs(self.fdm["attitude/theta-rad"])
        spin = abs(self.fdm["velocities/p-rad_sec"]) + abs(self.fdm["velocities/q-rad_sec"])
        jerk = float(np.sum(np.abs(action - self.prev_action)))

        reward = (
            1.0
            - self.reward_alt_weight * alt_err_ft
            - self.reward_heading_weight * progress * heading_err
            - self.reward_tilt_weight * tilt
            - self.reward_spin_weight * spin
            - self.reward_jerk_weight * jerk
        )

        crashed = self._is_crashed()
        if crashed:
            reward -= self.crash_penalty

        # DUZELTME: hedef irtifaya ulasip bir sure orada kalinca
        # (success_hold_steps) episode BASARIYLA sonlanir.
        if alt_err_ft < self.success_alt_tol_ft:
            self._success_counter += 1
        else:
            self._success_counter = 0

        reached = self._success_counter >= self.success_hold_steps
        if reached:
            reward += self.success_bonus

        info = self._get_telemetry(crashed, reached)

        self.prev_action = action.copy()

        terminated = bool(crashed or reached)
        truncated = bool(self.step_count >= self.max_steps)

        return obs, float(reward), terminated, truncated, info



Overwriting /content/repo/src/drone_rl/envs/f450_flight_env.py


In [29]:
%%writefile /content/repo/src/drone_rl/visualize.py
"""Egitilmis PPO politikasini calistirip 3D animasyon icin telemetri toplar."""

import json
import numpy as np
from stable_baselines3.common.vec_env import VecNormalize

from drone_rl.config import load_config
from drone_rl.env_factory import make_eval_vec_env, make_flight_eval_vec_env
from drone_rl.evaluate import resolve_model_paths, ALGO_CLASSES


def export_telemetry_json(telem, path):
    serializable = {}
    for k, v in telem.items():
        if hasattr(v, "tolist"):
            serializable[k] = v.tolist()
        else:
            serializable[k] = v
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f)
    print(f"Telemetri JSON kaydedildi: {path}")


def run_inference_episode(algo, run, config=None, task="hover", use_best=False, deterministic=True):
    """task='hover' (varsayilan, eski davranis) veya task='flight'."""
    cfg = load_config(config)
    from pathlib import Path
    run = Path(run)
    model_path, vecnorm_path = resolve_model_paths(run, use_best)

    if not vecnorm_path.exists():
        raise FileNotFoundError(f"VecNormalize dosyasi bulunamadi: {vecnorm_path}")

    if task == "hover":
        venv = make_eval_vec_env(cfg.env)
    else:
        venv = make_flight_eval_vec_env(cfg.flight_env)

    venv = VecNormalize.load(str(vecnorm_path), venv)
    venv.training = False
    venv.norm_reward = False

    model = ALGO_CLASSES[algo].load(str(model_path), device="cpu")
    raw = venv.envs[0]
    control_dt = raw.control_dt

    records = []
    obs = venv.reset()
    t = 0.0

    while True:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, _, done, infos = venv.step(action)
        telem = infos[0]

        # Aksiyon (-1..+1) yerine gercek motor gucunu (0..1) kaydediyoruz -
        # env'in kendi throttle donusum formulu ile ayni.
        motor_throttle = np.clip(
            raw.hover_throttle + action[0] * raw.throttle_range, 0.0, 1.0
        )

        rec = {
            "t": t,
            "x_m": telem["x_m"],
            "y_m": telem["y_m"],
            "alt_ft": telem["alt_ft"],
            "roll_rad": telem["roll_rad"],
            "pitch_rad": telem["pitch_rad"],
            "yaw_rad": telem["yaw_rad"],
            "alt_err_ft": telem["alt_err_ft"],
            "crashed": telem["crashed"],
            "motor_throttle": motor_throttle.tolist(),
        }
        if "target_heading_rad" in telem:
            rec["target_heading_rad"] = telem["target_heading_rad"]
            rec["along_track_fps"] = telem["along_track_fps"]
            rec["cross_track_fps"] = telem["cross_track_fps"]
        records.append(rec)
        t += control_dt
        if done[0]:
            break

    out = {k: np.array([r[k] for r in records]) for k in records[0].keys()}
    out["control_dt"] = control_dt
    out["target_altitude_ft"] = raw.target_altitude
    return out

Overwriting /content/repo/src/drone_rl/visualize.py


In [ ]:
!ls -la /content/runs/

ls: cannot access '/content/runs/': No such file or directory


In [9]:
%%writefile /content/repo/src/drone_rl/train.py
"""F450 ucus gorevleri icin PPO egitimi + EvalCallback.

--task hover  -> F450HoverEnv (sabit irtifa)
--task flight -> F450FlightEnv (rastgele irtifa + rastgele heading)
"""

import argparse
from pathlib import Path
import numpy as np
import torch.nn as nn
from drone_rl.acmi_writer import ACMIWriter
from drone_rl.utils.units import ft_to_m

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback

from drone_rl.config import load_config
from drone_rl.env_factory import (
    make_training_vec_env, make_flight_training_vec_env,
)


class SaveVecNormalizeCallback(BaseCallback):
    def __init__(self, save_path: Path):
        super().__init__()
        self.save_path = Path(save_path)

    def _on_step(self) -> bool:
        vec_normalize = self.model.get_vec_normalize_env()
        if vec_normalize is not None:
            self.save_path.mkdir(parents=True, exist_ok=True)
            vec_normalize.save(str(self.save_path / "vecnormalize_best.pkl"))
        return True


class ACMISnapshotCallback(BaseCallback):
    def __init__(self, eval_env, out_dir: Path, eval_freq: int, control_dt: float):
        super().__init__()
        self.eval_env = eval_env
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.eval_freq = eval_freq
        self.control_dt = control_dt

    def _on_step(self) -> bool:
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            self._record_snapshot()
        return True

    def _record_snapshot(self):
        acmi = ACMIWriter(name="F450", obj_type="Air+Rotorcraft+UAV", color="Blue")
        obs = self.eval_env.reset()
        t = 0.0

        while True:
            action, _ = self.model.predict(obs, deterministic=True)
            obs, _, done, infos = self.eval_env.step(action)
            telem = infos[0]

            acmi.add_frame(
                t=t,
                lon_deg=telem["lon_deg"],
                lat_deg=telem["lat_deg"],
                alt_m=ft_to_m(telem["alt_sl_ft"]),
                roll_deg=np.degrees(telem["roll_rad"]),
                pitch_deg=np.degrees(telem["pitch_rad"]),
                yaw_deg=np.degrees(telem["yaw_rad"]),
            )
            t += self.control_dt

            if done[0]:
                break

        path = self.out_dir / f"snapshot_{self.num_timesteps}.acmi"
        acmi.save(path)
        print(f"  [ACMI] snapshot kaydedildi: {path}", flush=True)


ACTIVATION_MAP = {"tanh": nn.Tanh, "relu": nn.ReLU}


def build_policy_kwargs(cfg_ppo):
    has_custom = (
        cfg_ppo.net_arch_pi is not None
        or cfg_ppo.net_arch_vf is not None
        or cfg_ppo.activation_fn is not None
    )
    if not has_custom:
        return None

    kwargs = {}
    pi_arch = cfg_ppo.net_arch_pi if cfg_ppo.net_arch_pi is not None else [64, 64]
    vf_arch = cfg_ppo.net_arch_vf if cfg_ppo.net_arch_vf is not None else [64, 64]
    kwargs["net_arch"] = dict(pi=pi_arch, vf=vf_arch)

    if cfg_ppo.activation_fn is not None:
        act_key = cfg_ppo.activation_fn.lower()
        if act_key not in ACTIVATION_MAP:
            raise ValueError(f"Bilinmeyen activation_fn: {cfg_ppo.activation_fn!r}")
        kwargs["activation_fn"] = ACTIVATION_MAP[act_key]

    return kwargs


def build_model(algo: str, cfg, venv, tensorboard_log: str):
    if algo != "ppo":
        raise ValueError(f"Bilinmeyen algoritma: {algo!r} (sadece ppo destekleniyor)")

    policy_kwargs = build_policy_kwargs(cfg.ppo)
    return PPO(
        cfg.ppo.policy, venv,
        n_steps=cfg.ppo.n_steps, batch_size=cfg.ppo.batch_size, n_epochs=cfg.ppo.n_epochs,
        gamma=cfg.ppo.gamma, gae_lambda=cfg.ppo.gae_lambda, clip_range=cfg.ppo.clip_range,
        learning_rate=cfg.ppo.learning_rate, ent_coef=cfg.ppo.ent_coef,
        policy_kwargs=policy_kwargs,
        verbose=1, device="cpu",
        tensorboard_log=tensorboard_log,
    )


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--algo", type=str, choices=["ppo"], default="ppo")
    ap.add_argument("--task", type=str, choices=["hover", "flight"], default="hover",
                     help="hover: sabit irtifa | flight: rastgele irtifa+heading")
    ap.add_argument("--config", type=str, default=None)
    ap.add_argument("--timesteps", type=int, default=None)
    ap.add_argument("--n-envs", type=int, default=None)
    ap.add_argument("--out", type=str, default="/content/runs/run")
    ap.add_argument("--eval-freq", type=int, default=10000)
    args = ap.parse_args()

    cfg = load_config(args.config)
    timesteps = args.timesteps if args.timesteps is not None else cfg.train.timesteps
    n_envs = args.n_envs if args.n_envs is not None else cfg.train.n_envs

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    if args.task == "hover":
        venv = make_training_vec_env(cfg.env, n_envs=n_envs, training=True, norm_reward=True)
        eval_env = make_training_vec_env(cfg.env, n_envs=1, training=False, norm_reward=False)
    else:
        venv = make_flight_training_vec_env(cfg.flight_env, n_envs=n_envs, training=True, norm_reward=True)
        eval_env = make_flight_training_vec_env(cfg.flight_env, n_envs=1, training=False, norm_reward=False)

    model = build_model(args.algo, cfg, venv, tensorboard_log=str(out / "tb"))

    ckpt_cb = CheckpointCallback(
        save_freq=max(20_000 // n_envs, 1),
        save_path=str(out / "ckpt"),
        name_prefix=args.algo,
    )

    best_model_path = out / "best_model"
    save_vecnorm_cb = SaveVecNormalizeCallback(best_model_path)

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=str(best_model_path),
        callback_on_new_best=save_vecnorm_cb,
        log_path=str(out / "logs"),
        eval_freq=max(args.eval_freq // n_envs, 1),
        deterministic=True,
        render=False,
    )

    raw_eval_env = eval_env.venv.envs[0].unwrapped
    acmi_snapshot_cb = ACMISnapshotCallback(
        eval_env=eval_env,
        out_dir=out / "acmi_snapshots",
        eval_freq=max(args.eval_freq // n_envs, 1),
        control_dt=raw_eval_env.control_dt,
    )

    model.learn(total_timesteps=timesteps, callback=[ckpt_cb, eval_cb, acmi_snapshot_cb])

    model.save(out / "model_final")
    venv.save(str(out / "vecnormalize.pkl"))
    print(f"Egitim tamamlandi ve kaydedildi ({args.algo}, task={args.task}):", out)
    print("  Son model      :", out / "model_final.zip", "+", out / "vecnormalize.pkl")
    print("  En iyi model   :", best_model_path / "best_model.zip", "+", best_model_path / "vecnormalize_best.pkl")


if __name__ == "__main__":
    main()



Overwriting /content/repo/src/drone_rl/train.py


In [ ]:
#ppo hover training
!cd /content/repo/src && python -m drone_rl.train --algo ppo --timesteps 600000 --n-envs 4 --out /content/repo/runs/hover_ppo_v1

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Faile

In [33]:
#kayıtlı telemetryden json oluşturucu
from drone_rl.visualize import run_inference_episode, export_telemetry_json

telem = run_inference_episode(
    algo="ppo",
    run="/content/repo/runs/flight_ppo_5sep",
    config="/content/repo/configs/ppo_flight.yaml",
    task="flight",
)
export_telemetry_json(telem, "/content/repo/telemetry_for_html.json")

from google.colab import files
files.download('/content/repo/telemetry_for_html.json')


Telemetri JSON kaydedildi: /content/repo/telemetry_for_html.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
#eğer eval.py den dolayı motor verileri gelmezse çalıştır tekrar telemetry ve json oluştur.
import importlib
import drone_rl.config, drone_rl.env_factory, drone_rl.evaluate, drone_rl.visualize

importlib.reload(drone_rl.config)
importlib.reload(drone_rl.env_factory)
importlib.reload(drone_rl.evaluate)
importlib.reload(drone_rl.visualize)

from drone_rl.visualize import run_inference_episode, export_telemetry_json

In [18]:
#ppo flight eğitim
!cd /content/repo/src && python -m drone_rl.train --task flight --config ../configs/ppo_flight.yaml --out /content/repo/runs/flight_ppo_5sep

2026-09-05 11:20:43.384616: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-05 11:20:43.644530: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT.

In [32]:
#fligth modeli için eval ve acmi kaydedici
!cd /content/repo/src && python -m drone_rl.evaluate \
    --algo ppo --task flight \
    --config ../configs/ppo_flight.yaml \
    --run /content/repo/runs/flight_ppo_5sep \
    --output /content/repo/ppo_flight_telemetry.csv \
    --acmi-output /content/repo/ppo_flight_final.acmi \
    --episodes 5


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Faile

In [ ]:

#sac eğitim hover için başlat
#!cd /content/repo/src && python -m drone_rl.train --algo sac --config ../configs/ppo_hover.yaml --timesteps 600000 --n-envs 4 --out /content/runs/hover_sac_v1

#ppo ve sac için ayrı ayrı hover evaluation ama last modeli alır
#!cd /content/repo/src && python -m drone_rl.evaluate --algo ppo --config ../configs/ppo_hover.yaml --run /content/runs/hover_ppo_v1 --output /content/ppo_telemetry.csv --acmi-output /content/ppo_final.acmi --episodes 3
#!cd /content/repo/src && python -m drone_rl.evaluate --algo sac --config ../configs/ppo_hover.yaml --run /content/runs/hover_sac_v1 --output /content/sac_telemetry.csv --acmi-output /content/sac_final.acmi --episodes 3

#yine ppo ve sac için hover evaluation kayıtları ama best model kullanır last değil
#!cd /content/repo/src && python -m drone_rl.evaluate --algo ppo --config ../configs/ppo_hover.yaml --run /content/runs/hover_ppo_v1 --output /content/ppo_best_telemetry.csv --episodes 5 --use-best
#!cd /content/repo/src && python -m drone_rl.evaluate --algo sac --config ../configs/ppo_hover.yaml --run /content/runs/hover_sac_v1 --output /content/sac_best_telemetry.csv --episodes 5 --use-best

#hyperparameter optimization via optuna
#!cd /content/repo/src && python -m drone_rl.tune --algo ppo --n-trials 20 --timesteps-per-trial 60000 --out /content/tuning/ppo
#!cd /content/repo/src && python -m drone_rl.tune --algo sac --n-trials 20 --timesteps-per-trial 60000 --out /content/tuning/sac

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Faile

In [ ]:
#optuna ile optimize edilen parametreler
!cat /content/tuning/ppo/best_params_ppo.json

{
  "algo": "ppo",
  "best_value": 333.0058974,
  "best_params": {
    "n_steps": 1024,
    "learning_rate": 0.0009682086811397772,
    "batch_size": 256,
    "n_epochs": 8,
    "gamma": 0.98,
    "gae_lambda": 0.821733218323567,
    "clip_range": 0.30028032106674823,
    "ent_coef": 0.008527398284507002
  }
}

In [22]:
%%writefile /content/repo/src/drone_rl/evaluate.py
"""Egitilmis PPO politikasini calistir; CSV telemetri ve/veya
gercek ACMI (Tacview) dosyasi olarak kaydet.

--task hover  -> F450HoverEnv
--task flight -> F450FlightEnv
"""

import argparse
from pathlib import Path
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize

from drone_rl.config import load_config
from drone_rl.env_factory import make_eval_vec_env, make_flight_eval_vec_env
from drone_rl.utils.units import ft_to_m
from drone_rl.acmi_writer import ACMIWriter


ALGO_CLASSES = {"ppo": PPO}


def resolve_model_paths(run: Path, use_best: bool):
    if use_best:
        return run / "best_model" / "best_model", run / "best_model" / "vecnormalize_best.pkl"
    return run / "model_final", run / "vecnormalize.pkl"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--algo", type=str, choices=["ppo"], default="ppo")
    ap.add_argument("--task", type=str, choices=["hover", "flight"], default="hover")
    ap.add_argument("--config", type=str, default=None)
    ap.add_argument("--run", type=str, default="/content/repo/runs/run")
    ap.add_argument("--episodes", type=int, default=3)
    ap.add_argument("--output", type=str, default="/content/telemetry.csv")
    ap.add_argument("--acmi-output", type=str, default=None)
    ap.add_argument("--use-best", action="store_true")
    args = ap.parse_args()

    cfg = load_config(args.config)
    run = Path(args.run)
    model_path, vecnorm_path = resolve_model_paths(run, args.use_best)

    if not vecnorm_path.exists():
        raise FileNotFoundError(f"VecNormalize dosyasi bulunamadi: {vecnorm_path}")

    if args.task == "hover":
        venv = make_eval_vec_env(cfg.env)
    else:
        venv = make_flight_eval_vec_env(cfg.flight_env)

    venv = VecNormalize.load(str(vecnorm_path), venv)
    venv.training = False
    venv.norm_reward = False

    algo_cls = ALGO_CLASSES[args.algo]
    model = algo_cls.load(str(model_path), device="cpu")
    raw = venv.envs[0]
    control_dt = raw.control_dt

    all_telemetry = []
    acmi = ACMIWriter(name="F450", obj_type="Air+Rotorcraft+UAV", color="Blue") \
        if args.acmi_output else None

    global_t = 0.0

    for ep in range(args.episodes):
        obs = venv.reset()
        t = 0.0

        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, infos = venv.step(action)
            telem = infos[0]

            # Aksiyon (-1..+1) yerine gercek motor gucunu (0..1) kaydediyoruz -
            # env'in kendi throttle donusum formulu ile ayni.
            motor_throttle = np.clip(
                raw.hover_throttle + action[0] * raw.throttle_range, 0.0, 1.0
            )

            data = {
                "timestamp": round(t, 4),
                "episode": ep,
                "alt_ft": round(telem["alt_ft"], 4),
                "roll_rad": round(telem["roll_rad"], 6),
                "pitch_rad": round(telem["pitch_rad"], 6),
                "yaw_rad": round(telem["yaw_rad"], 6),
                "alt_err_ft": round(telem["alt_err_ft"], 4),
                "crashed": telem["crashed"],
                "motor_1": round(float(motor_throttle[0]), 4),
                "motor_2": round(float(motor_throttle[1]), 4),
                "motor_3": round(float(motor_throttle[2]), 4),
                "motor_4": round(float(motor_throttle[3]), 4),
            }
            if "target_heading_rad" in telem:
                data["target_heading_rad"] = round(telem["target_heading_rad"], 4)
                data["along_track_fps"] = round(telem["along_track_fps"], 4)
                data["cross_track_fps"] = round(telem["cross_track_fps"], 4)
            if "reached_target" in telem:
                data["reached_target"] = telem["reached_target"]
            all_telemetry.append(data)

            if acmi is not None:
                alt_m = ft_to_m(telem["alt_sl_ft"])
                acmi.add_frame(
                    t=global_t,
                    lon_deg=telem["lon_deg"],
                    lat_deg=telem["lat_deg"],
                    alt_m=alt_m,
                    roll_deg=np.degrees(telem["roll_rad"]),
                    pitch_deg=np.degrees(telem["pitch_rad"]),
                    yaw_deg=np.degrees(telem["yaw_rad"]),
                )

            t += control_dt
            global_t += control_dt
            if done[0]:
                break

    df = pd.DataFrame(all_telemetry)
    df.to_csv(args.output, index=False, header=True)
    print(f"CSV telemetri kaydedildi: {args.output}")
    print(df.head())

    if acmi is not None:
        saved_path = acmi.save(args.acmi_output)
        print(f"ACMI (Tacview) dosyasi kaydedildi: {saved_path}")


if __name__ == '__main__':
    main()


Overwriting /content/repo/src/drone_rl/evaluate.py


In [11]:
#gitignore oluşturur ve günceller
with open("/content/repo/.gitignore", "w") as f:
    f.write("__pycache__/\n.ipynb_checkpoints/\nruns/*/tb/\nruns/*/ckpt/\n")
!cat /content/repo/.gitignore

__pycache__/
.ipynb_checkpoints/
runs/*/tb/
runs/*/ckpt/


In [12]:
%%writefile /content/repo/configs/ppo_flight.yaml
flight_env:
  target_altitude_min_ft: 20.0
  target_altitude_max_ft: 45.0
  target_speed_fps: 6.0
  episode_seconds: 60.0
  control_hz: 20
  physics_hz: 240
  hover_throttle: 0.420
  throttle_range: 0.25
  reward_alt_weight: 0.10
  reward_heading_weight: 0.08
  reward_tilt_weight: 0.05
  reward_spin_weight: 0.10
  reward_jerk_weight: 0.05
  # --- YENI: tirmanma baslangici + hedefe ulasinca reset (success) ---
  altitude_start_offset_ft: 25.0
  altitude_start_jitter_ft: 2.0
  success_alt_tol_ft: 1.5
  success_hold_seconds: 1.0
  success_bonus: 20.0
ppo:
  policy: MlpPolicy
  n_steps: 1024
  batch_size: 256
  n_epochs: 8
  gamma: 0.98
  gae_lambda: 0.821733218323567
  clip_range: 0.30028032106674823
  learning_rate: 0.0009682086811397772
  ent_coef: 0.008527398284507002
  net_arch_pi: [128, 128]
  net_arch_vf: [128, 128]
  activation_fn: tanh
train:
  timesteps: 600000
  n_envs: 4

Overwriting /content/repo/configs/ppo_flight.yaml


In [ ]:
readme = """# quadcopter-rl-copilot

JSBSim F450 quadcopter modeli uzerinde PPO ile hover kontrolu.

## Sonuc (hover_v1)

300.000 adim egitim sonrasi, 5 degerlendirme episode'unda:

- Episode uzunlugu: 400/400 (hic dusme yok)
- Hedef irtifadan ortalama sapma: 0.02 - 0.06 ft
- Ortalama egilme: 0.02 - 0.08 rad

## Kurulum (Colab)

    !pip install -q stable-baselines3 gymnasium
    !pip uninstall -y -q jsbsim
    !pip install -q jsbsim==1.2.4

Repoyu klonladiktan sonra src dizinini Python yoluna ekle:

    import os, sys
    sys.path.insert(0, "/content/repo/src")
    os.environ["PYTHONPATH"] = "/content/repo/src"

## Kullanim

Egitim:

    cd src && python -m drone_rl.train --timesteps 300000 --n-envs 4 --out ../runs/hover_v2

Degerlendirme:

    cd src && python -m drone_rl.evaluate --run ../runs/hover_v1 --csv /content/iz.csv

## Yapi

- src/drone_rl/envs/f450_env.py - Gymnasium ortami
- src/drone_rl/train.py - PPO egitimi
- src/drone_rl/evaluate.py - egitilmis politikanin olculmesi
- configs/ppo_hover.yaml - kullanilan ayarlar
- runs/hover_v1/ - egitilmis model ve normalizasyon istatistikleri
- notebooks/quadcopter_rl.ipynb - Colab calisma defteri

## Notlar

- JSBSim emperyal birim kullanir (ft, lbs, fps).
- Hover gazi 0.410 olarak olculdu. Aksiyon bu deger etrafinda +-0.25
  araliginda olceklenir, boylece sifir aksiyon "asili kal" anlamina gelir.
- vecnormalize.pkl model ile birlikte yuklenmelidir, aksi halde politika
  yanlis olcekli gozlem alir ve calismaz.
- F450 XML'i yuklenirken "version 3.0" uyarisi verir; zararsizdir.
"""

with open("/content/repo/README.md", "w") as f:
    f.write(readme)

print(open("/content/repo/README.md").read()[:300])

# quadcopter-rl-copilot

JSBSim F450 quadcopter modeli uzerinde PPO ile hover kontrolu.

## Sonuc (hover_v1)

300.000 adim egitim sonrasi, 5 degerlendirme episode'unda:

- Episode uzunlugu: 400/400 (hic dusme yok)
- Hedef irtifadan ortalama sapma: 0.02 - 0.06 ft
- Ortalama egilme: 0.02 - 0.08 rad

#


In [13]:
%%writefile /content/repo/src/drone_rl/acmi_writer.py
"""Tacview ACMI (.acmi) format yazici - basit tek-obje coklu-episode kaydi.

ACMI format referansi: https://www.tacview.net/documentation/acmi/en/

Bu writer sadece bu proje icin gerekli minimum alt kumeyi destekler:
tek bir hava araci objesi, zaman serisi konum/aci guncellemeleri.
Coklu obje, olay (event) kayitlari, veya ek property'ler (hiz, RPM vb.)
desteklenmiyor - ihtiyac olursa genisletilebilir.

Beklenen birimler (ACMI standardi):
- lon_deg, lat_deg : derece (WGS84)
- alt_m            : metre (deniz seviyesinden veya yerden - tutarli olmasi yeterli)
- roll_deg, pitch_deg, yaw_deg : derece

JSBSim ft ve radyan kullandigi icin cagiran kod bu donusumu
(units.ft_to_m, np.degrees) yapmali; bu siniftan once cagirilmalidir.
"""

from pathlib import Path
from datetime import datetime, timezone


class ACMIWriter:
    def __init__(self, object_id=1, name="F450", obj_type="Air+Rotorcraft+UAV", color="Blue"):
        self.object_id = object_id
        self.name = name
        self.obj_type = obj_type
        self.color = color
        self._lines = []
        self._header_written = False
        self._object_declared = False
        self._last_frame_time = None

    def _write_header(self):
        now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        self._lines.append("FileType=text/acmi/tacview")
        self._lines.append("FileVersion=2.2")
        self._lines.append(f"0,ReferenceTime={now}")
        self._header_written = True

    def add_frame(self, t, lon_deg, lat_deg, alt_m, roll_deg, pitch_deg, yaw_deg):
        """Bir zaman anindaki obje durumunu ekler.

        t: saniye cinsinden, dosya boyunca MONOTONIK ARTAN olmali
           (birden fazla episode varsa t'yi sifirlama, devam ettir).
        """
        if not self._header_written:
            self._write_header()

        # Ayni t icin tekrar frame acmayalim (float hassasiyeti icin yuvarla)
        t_rounded = round(t, 2)
        if self._last_frame_time != t_rounded:
            self._lines.append(f"#{t_rounded:.2f}")
            self._last_frame_time = t_rounded

        obj_line = (
            f"{self.object_id:x},T={lon_deg:.7f}|{lat_deg:.7f}|{alt_m:.2f}|"
            f"{roll_deg:.2f}|{pitch_deg:.2f}|{yaw_deg:.2f}"
        )
        if not self._object_declared:
            obj_line += f",Name={self.name},Type={self.obj_type},Color={self.color}"
            self._object_declared = True

        self._lines.append(obj_line)

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write("\n".join(self._lines) + "\n")
        return path


Overwriting /content/repo/src/drone_rl/acmi_writer.py


In [14]:
%%writefile /content/repo/src/drone_rl/config.py
"""Merkezi config yukleme. YAML dosyasindaki degerleri dataclass'lara donusturur."""

from dataclasses import dataclass, field
from typing import Optional, List
import yaml


@dataclass
class EnvConfig:
    """F450HoverEnv icin ayarlar (hedef irtifa sabit)."""
    target_altitude_ft: float = 30.0
    episode_seconds: float = 20.0
    control_hz: int = 20
    physics_hz: int = 240
    hover_throttle: float = 0.420
    throttle_range: float = 0.25
    reward_alt_weight: float = 0.10
    reward_tilt_weight: float = 0.50
    reward_spin_weight: float = 0.10
    reward_jerk_weight: float = 0.05
    crash_penalty: float = 50.0
    crash_min_alt_ft: float = 1.0
    crash_max_alt_offset_ft: float = 60.0
    crash_max_tilt_rad: float = 1.0


@dataclass
class FlightEnvConfig:
    """F450FlightEnv icin ayarlar (hedef irtifa + hedef yon, ikisi de
    her episode'da rastgele). EnvConfig'ten BAGIMSIZ, ayri bir dataclass -
    hover config'ini hic etkilemez.

    DUZELTME: drone artik hedef irtifanin altitude_start_offset_ft kadar
    ALTINDAN spawn oluyor (gercek bir tirmanma yasaniyor) ve hedef irtifaya
    ulasip success_hold_seconds kadar orada kalinca episode basariyla
    (success_bonus ile) sonlaniyor - artik sadece sure dolunca degil.
    """
    target_altitude_min_ft: float = 20.0
    target_altitude_max_ft: float = 45.0
    target_speed_fps: float = 6.0
    episode_seconds: float = 60.0
    control_hz: int = 20
    physics_hz: int = 240
    hover_throttle: float = 0.420
    throttle_range: float = 0.25
    reward_alt_weight: float = 0.10
    reward_heading_weight: float = 0.08
    reward_tilt_weight: float = 0.05
    reward_spin_weight: float = 0.10
    reward_jerk_weight: float = 0.05
    crash_penalty: float = 50.0
    crash_min_alt_ft: float = 1.0
    crash_max_alt_offset_ft: float = 60.0
    crash_max_tilt_rad: float = 1.0
    # --- YENI: tirmanma baslangici + basari (success) ayarlari ---
    altitude_start_offset_ft: float = 25.0
    altitude_start_jitter_ft: float = 2.0
    success_alt_tol_ft: float = 1.5
    success_hold_seconds: float = 1.0
    success_bonus: float = 20.0


@dataclass
class PPOConfig:
    policy: str = "MlpPolicy"
    n_steps: int = 1024
    batch_size: int = 256
    n_epochs: int = 10
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_range: float = 0.2
    learning_rate: float = 3e-4
    ent_coef: float = 0.0
    net_arch_pi: Optional[List[int]] = None
    net_arch_vf: Optional[List[int]] = None
    activation_fn: Optional[str] = None


@dataclass
class TrainConfig:
    timesteps: int = 300_000
    n_envs: int = 4


@dataclass
class Config:
    env: EnvConfig = field(default_factory=EnvConfig)
    flight_env: FlightEnvConfig = field(default_factory=FlightEnvConfig)
    ppo: PPOConfig = field(default_factory=PPOConfig)
    train: TrainConfig = field(default_factory=TrainConfig)


def load_config(path: Optional[str]) -> Config:
    if path is None:
        return Config()

    with open(path, "r") as f:
        raw = yaml.safe_load(f) or {}

    return Config(
        env=EnvConfig(**raw.get("env", {})),
        flight_env=FlightEnvConfig(**raw.get("flight_env", {})),
        ppo=PPOConfig(**raw.get("ppo", {})),
        train=TrainConfig(**raw.get("train", {})),
    )




Overwriting /content/repo/src/drone_rl/config.py


In [15]:
%%writefile /content/repo/src/drone_rl/tune.py
"""Optuna ile PPO icin hiperparametre optimizasyonu.

Bu, train.py'nin bir 'modu' degil, ayri bir arac: her calisma (trial)
kisa bir egitim yapip sonucu (ortalama reward) Optuna'ya bildiriyor,
Optuna da bir sonraki denemede hangi hiperparametreleri deneyecegini
bu geri bildirime gore seciyor.

Kullanim:
    python -m drone_rl.tune --algo ppo --n-trials 30 --timesteps-per-trial 60000 --out /content/tuning/ppo
"""

import argparse
import json
from pathlib import Path

import optuna
from optuna.pruners import MedianPruner
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback

from drone_rl.config import load_config
from drone_rl.env_factory import make_training_vec_env


class TrialEvalCallback(EvalCallback):
    """Normal EvalCallback gibi periyodik degerlendirme yapar, ama
    her degerlendirme sonucunu Optuna'ya da raporlar VE ekrana bir
    ilerleme satiri yazdirir."""

    def __init__(self, eval_env, trial, total_timesteps, **kwargs):
        super().__init__(eval_env, **kwargs)
        self.trial = trial
        self.total_timesteps = total_timesteps
        self.eval_idx = 0

    def _on_step(self) -> bool:
        continue_training = super()._on_step()
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            self.eval_idx += 1
            print(
                f"  [trial {self.trial.number}] "
                f"{self.num_timesteps}/{self.total_timesteps} adim - "
                f"ortalama reward: {self.last_mean_reward:.2f}",
                flush=True,
            )
            self.trial.report(self.last_mean_reward, self.eval_idx)
            if self.trial.should_prune():
                print(f"  [trial {self.trial.number}] erken kesildi (prune)", flush=True)
                raise optuna.TrialPruned()
        return continue_training


def suggest_ppo_params(trial: optuna.Trial) -> dict:
    """PPO icin arama uzayi. Yaygin/etkili PPO hiperparametreleri."""
    n_steps = trial.suggest_categorical("n_steps", [512, 1024, 2048])
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
        "n_steps": n_steps,
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
        "n_epochs": trial.suggest_int("n_epochs", 3, 30),
        "gamma": trial.suggest_categorical("gamma", [0.9, 0.95, 0.98, 0.99, 0.995, 0.999]),
        "gae_lambda": trial.suggest_float("gae_lambda", 0.8, 1.0),
        "clip_range": trial.suggest_float("clip_range", 0.1, 0.4),
        "ent_coef": trial.suggest_float("ent_coef", 1e-8, 0.1, log=True),
    }


def build_trial_model(algo: str, params: dict, venv):
    if algo != "ppo":
        raise ValueError(f"Bilinmeyen algoritma: {algo!r} (sadece ppo destekleniyor)")
    return PPO(
        "MlpPolicy", venv,
        n_steps=params["n_steps"], batch_size=params["batch_size"],
        n_epochs=params["n_epochs"], gamma=params["gamma"],
        gae_lambda=params["gae_lambda"], clip_range=params["clip_range"],
        learning_rate=params["learning_rate"], ent_coef=params["ent_coef"],
        verbose=0, device="cpu",
    )


def make_objective(algo: str, cfg, args):
    def objective(trial: optuna.Trial) -> float:
        params = suggest_ppo_params(trial)

        print(f"\n=== Trial {trial.number} basladi ===", flush=True)
        print(f"  Parametreler: {params}", flush=True)

        venv = make_training_vec_env(cfg.env, n_envs=args.n_envs, training=True, norm_reward=True)
        eval_env = make_training_vec_env(cfg.env, n_envs=1, training=False, norm_reward=False)

        model = build_trial_model(algo, params, venv)

        eval_cb = TrialEvalCallback(
            eval_env,
            trial=trial,
            total_timesteps=args.timesteps_per_trial,
            eval_freq=max(args.eval_freq // args.n_envs, 1),
            deterministic=True,
            render=False,
            verbose=0,
        )

        try:
            model.learn(total_timesteps=args.timesteps_per_trial, callback=eval_cb)
        except optuna.TrialPruned:
            venv.close()
            eval_env.close()
            raise

        reward = eval_cb.last_mean_reward
        venv.close()
        eval_env.close()

        print(f"=== Trial {trial.number} bitti - sonuc: {reward:.2f} ===", flush=True)

        if reward is None or reward != reward:
            return -1e6
        return float(reward)

    return objective


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--algo", type=str, choices=["ppo"], default="ppo",
                     help="Hangi algoritma icin hiperparametre aranacak (su an sadece ppo)")
    ap.add_argument("--config", type=str, default=None,
                     help="Ortam ayarlari icin bir config dosyasi "
                          "(sadece env: bolumu kullanilir)")
    ap.add_argument("--n-trials", type=int, default=30)
    ap.add_argument("--timesteps-per-trial", type=int, default=60_000)
    ap.add_argument("--n-envs", type=int, default=4)
    ap.add_argument("--eval-freq", type=int, default=10_000)
    ap.add_argument("--out", type=str, default="/content/tuning/result")
    ap.add_argument("--study-name", type=str, default=None)
    ap.add_argument("--storage", type=str, default=None,
                     help="Optuna icin kalici depolama (ornek: sqlite:////content/tuning/study.db).")
    args = ap.parse_args()

    optuna.logging.set_verbosity(optuna.logging.INFO)

    cfg = load_config(args.config)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=1)
    study = optuna.create_study(
        direction="maximize",
        pruner=pruner,
        study_name=args.study_name or f"{args.algo}_flight",
        storage=args.storage,
        load_if_exists=True,
    )

    print(f"Arama basliyor: algo={args.algo}, n_trials={args.n_trials}, "
          f"timesteps_per_trial={args.timesteps_per_trial}", flush=True)

    study.optimize(make_objective(args.algo, cfg, args), n_trials=args.n_trials)

    print("\n=== Arama tamamlandi ===")
    print("En iyi deger (ortalama reward):", study.best_value)
    print("En iyi parametreler:", study.best_params)

    best_path = out / f"best_params_{args.algo}.json"
    with open(best_path, "w") as f:
        json.dump({"algo": args.algo, "best_value": study.best_value,
                    "best_params": study.best_params}, f, indent=2)
    print("Kaydedildi:", best_path)

    trials_csv = out / f"trials_{args.algo}.csv"
    study.trials_dataframe().to_csv(trials_csv, index=False)
    print("Tum denemeler:", trials_csv)


if __name__ == "__main__":
    main()




Overwriting /content/repo/src/drone_rl/tune.py


In [16]:
%%writefile /content/repo/src/drone_rl/env_factory.py
"""Egitim ve degerlendirme icin ortak F450 ortami/VecEnv kurulum yardimcilari.

Hem F450HoverEnv (hover gorevi) hem F450FlightEnv (irtifa+heading gorevi)
icin ayri fonksiyon setleri barindirir. Biri digerini etkilemez.
"""

from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from drone_rl.envs.f450_env import F450HoverEnv
from drone_rl.envs.f450_flight_env import F450FlightEnv
from drone_rl.config import EnvConfig, FlightEnvConfig


# ---------------------------------------------------------------------
# Hover gorevi (degismedi)
# ---------------------------------------------------------------------

def make_env(env_config: EnvConfig) -> F450HoverEnv:
    return F450HoverEnv(
        target_altitude_ft=env_config.target_altitude_ft,
        episode_seconds=env_config.episode_seconds,
        physics_hz=env_config.physics_hz,
        control_hz=env_config.control_hz,
        hover_throttle=env_config.hover_throttle,
        throttle_range=env_config.throttle_range,
        reward_alt_weight=env_config.reward_alt_weight,
        reward_tilt_weight=env_config.reward_tilt_weight,
        reward_spin_weight=env_config.reward_spin_weight,
        reward_jerk_weight=env_config.reward_jerk_weight,
        crash_penalty=env_config.crash_penalty,
        crash_min_alt_ft=env_config.crash_min_alt_ft,
        crash_max_alt_offset_ft=env_config.crash_max_alt_offset_ft,
        crash_max_tilt_rad=env_config.crash_max_tilt_rad,
    )


def make_training_vec_env(env_config: EnvConfig, n_envs: int, training: bool,
                           norm_reward: bool, clip_obs: float = 10.0):
    def _make():
        return Monitor(make_env(env_config))

    venv = DummyVecEnv([_make for _ in range(n_envs)])
    venv = VecNormalize(
        venv, norm_obs=True, norm_reward=norm_reward, clip_obs=clip_obs, training=training
    )
    return venv


def make_eval_vec_env(env_config: EnvConfig):
    return DummyVecEnv([lambda: make_env(env_config)])


# ---------------------------------------------------------------------
# Flight gorevi (hedef irtifa + hedef yon; artik tirmanma + success reset)
# ---------------------------------------------------------------------

def make_flight_env(flight_config: FlightEnvConfig) -> F450FlightEnv:
    return F450FlightEnv(
        target_altitude_min_ft=flight_config.target_altitude_min_ft,
        target_altitude_max_ft=flight_config.target_altitude_max_ft,
        target_speed_fps=flight_config.target_speed_fps,
        episode_seconds=flight_config.episode_seconds,
        physics_hz=flight_config.physics_hz,
        control_hz=flight_config.control_hz,
        hover_throttle=flight_config.hover_throttle,
        throttle_range=flight_config.throttle_range,
        reward_alt_weight=flight_config.reward_alt_weight,
        reward_heading_weight=flight_config.reward_heading_weight,
        reward_tilt_weight=flight_config.reward_tilt_weight,
        reward_spin_weight=flight_config.reward_spin_weight,
        reward_jerk_weight=flight_config.reward_jerk_weight,
        crash_penalty=flight_config.crash_penalty,
        crash_min_alt_ft=flight_config.crash_min_alt_ft,
        crash_max_alt_offset_ft=flight_config.crash_max_alt_offset_ft,
        crash_max_tilt_rad=flight_config.crash_max_tilt_rad,
        # --- YENI ---
        altitude_start_offset_ft=flight_config.altitude_start_offset_ft,
        altitude_start_jitter_ft=flight_config.altitude_start_jitter_ft,
        success_alt_tol_ft=flight_config.success_alt_tol_ft,
        success_hold_seconds=flight_config.success_hold_seconds,
        success_bonus=flight_config.success_bonus,
    )


def make_flight_training_vec_env(flight_config: FlightEnvConfig, n_envs: int,
                                  training: bool, norm_reward: bool, clip_obs: float = 10.0):
    def _make():
        return Monitor(make_flight_env(flight_config))

    venv = DummyVecEnv([_make for _ in range(n_envs)])
    venv = VecNormalize(
        venv, norm_obs=True, norm_reward=norm_reward, clip_obs=clip_obs, training=training
    )
    return venv


def make_flight_eval_vec_env(flight_config: FlightEnvConfig):
    return DummyVecEnv([lambda: make_flight_env(flight_config)])


Overwriting /content/repo/src/drone_rl/env_factory.py


In [3]:
%%bash
# Repo dizinine geç
cd /content/repo

# Tüm değişiklikleri ekle
git add .

git commit -m "little visualization changes"
git pull origin main --no-edit
git push origin main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Already up to date.


From https://github.com/miray7yuce/quadcopter-rl-copilot
 * branch            main       -> FETCH_HEAD
Everything up-to-date


In [ ]:
from google.colab import files
files.download('/content/ppo_telemetry.csv')
files.download('/content/ppo_final.acmi')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>